In [3]:
import importlib
import asyncio
import os
from dotenv import load_dotenv

# Reload leaf modules before modules that import from them. This avoids mixed
# class generations when iterating in a notebook after editing source files.
RELOAD_MODULES = [
    # Generic utilities and CCRS foundations.
    "react_agent.utils.settings",
    "react_agent.utils.logging_config",
    "react_agent.ccrs.audit",
    "react_agent.ccrs.rdf_adapter",
    "react_agent.ccrs.java_logging",
    "react_agent.ccrs.java_runtime",
    "react_agent.ccrs.state",

    # Contingency leaf modules first; importing modules follow later.
    "react_agent.ccrs.contingency.situation",
    "react_agent.ccrs.contingency.in_memory_ccrs_trace_history",
    "react_agent.ccrs.contingency.interaction",
    "react_agent.ccrs.contingency.ccrs_context",
    "react_agent.ccrs.contingency.contingency_ccrs_result",
    "react_agent.ccrs.contingency.opportunistic_guidance",
    "react_agent.ccrs.contingency.contingency_ccrs",
    "react_agent.ccrs.contingency.escalation",
    "react_agent.ccrs.contingency.default_escalation_controller",
    "react_agent.ccrs.prompt",

    # Opportunistic CCRS modules.
    "react_agent.ccrs.opportunistic.vocabulary_matcher",
    "react_agent.ccrs.opportunistic.opportunistic_result",
    "react_agent.ccrs.opportunistic",

    # Prompt context imports the CCRS result-selection helpers above.
    "react_agent.ccrs.prompt_context",

    # Tools and graph nodes.
    "react_agent.tools.get",
    "react_agent.tools.post",
    "react_agent.tools",
    "react_agent.prompts.user_query",
    "react_agent.prompts.react_prompt",
    "react_agent.nodes.decision_node",
    "react_agent.ccrs.contingency.decision",
    "react_agent.ccrs.ccrs_node",
    "react_agent.ccrs.contingency",
    "react_agent.ccrs",
    "react_agent.nodes.llm_node",
    "react_agent.nodes.llm_node_ccrs_v2",
    "react_agent.nodes.tool_node",

    # Graphs and launch API last, after all imports they close over are fresh.
    "react_agent.graph.graph",
    "react_agent.graph.graph_ccrs",
    "react_agent.runner",
    "react_agent.api",
]

for module_name in RELOAD_MODULES:
    module = importlib.import_module(module_name)
    importlib.reload(module)

from react_agent.api import launch_agent
from react_agent.prompts.user_query import USER_QUERY, USER_QUERY_CCRS

load_dotenv(dotenv_path=".env", override=True)


True

### Basline React Agent (no CCRS)

In [4]:
await launch_agent(
    query=USER_QUERY,
    agent_name="react_baseline_v1_01",
    graph_name="graph",
    run_mode="async",
    log_level="INFO"
)

================================ Human Message =================================

You are an agent navigating an HTTP/RDF maze.

# GENERAL INSTRUCTIONS:

## Goal:
Reach the maze exit.

## Tools:
- Use http_get with exactly this argument shape: {"url": "..."}.
- Use http_post with exactly this argument shape: {"url": "...", "data": "...", "headers": {"Content-Type": "text/turtle"}}.
- In http_post, the target URI goes in the url argument. Do not put the target URI as a standalone line in data.
- In http_post, the data argument contains only the Turtle body triples.
- Every http_post call MUST include a non-empty data field.
- POST bodies must be valid Turtle.

## Environment principles:
1. The maze is made of cells. 
    - Each cell has a unique URI. 
    - Dereferencing a cell URI with http_get returns RDF describing that cell, including adjacent cells and possible interactions.
2. Embodiment: You are always located in a single current cell.
    - Embodiment enforces: 
        a) local

CancelledError: 

### React Agent with CCRS

In [2]:
from react_agent.utils.settings import settings

CONTINGENCY_CONFIGURATION = {
    "prediction_llm": {
        "max_history_actions": 100,
    },
}

await launch_agent(
    query=USER_QUERY_CCRS,
    agent_name="react_ccrs_mazeV1_04",
    graph_name="graph_ccrs",
    run_mode="async",
    log_level="INFO",
    enable_contingency_escalation_tool=True,
    enable_contingency_llm_prediction=True,
    enable_contingency_a2a_consultation=True,
    contingency_configuration=CONTINGENCY_CONFIGURATION,
)

================================ Human Message =================================

You are an agent navigating an HTTP/RDF maze.

# GENERAL INSTRUCTIONS:

## Goal:
Reach the maze exit.

## Tools:
- Use http_get with exactly this argument shape: {"url": "..."}.
- Use http_post with exactly this argument shape: {"url": "...", "data": "...", "headers": {"Content-Type": "text/turtle"}}.
- In http_post, the target URI goes in the url argument. Do not put the target URI as a standalone line in data.
- In http_post, the data argument contains only the Turtle body triples.
- Every http_post call MUST include a non-empty data field.
- POST bodies must be valid Turtle.

- Use the escalate_to_contingency_ccrs tool to trigger the execution of contingency Course Check and Revision Strategies (CCRS).
- Escalate to contingency CCRS only when you are about to fail, do not know how to proceed, are confused, or have been making no progress. 

## Environment principles:
1. The maze is made of cells. 
    